In [20]:
import os
print(os.path.exists("../data/toronto/Major_Crime_Indicators_Open_Data_-4289692410590149445.csv"))

True


In [21]:
import pandas as pd

#Toronto
toronto = pd.read_csv("../data/toronto/Major_Crime_Indicators_Open_Data_-4289692410590149445.csv")

#Vancouver
vancouver = pd.read_csv("../data/vancouver/crimedata_csv_AllNeighbourhoods_AllYears.csv")

In [22]:
# Toronto - Data cleaning
toronto_clean = toronto[[
    'OCC_YEAR', 'OCC_MONTH', 'OCC_DOW', 'OCC_HOUR',
    'OFFENCE', 'CSI_CATEGORY',
    'NEIGHBOURHOOD_158', 'PREMISES_TYPE', 'LOCATION_TYPE',
    'LAT_WGS84', 'LONG_WGS84'
]].copy()

# Change names for consistency
toronto_clean.columns = [
    'year', 'month', 'day_of_week', 'hour',
    'offence', 'crime_type',
    'neighbourhood', 'premise_type', 'location_type',
    'latitude', 'longitude'
]

# Toronto - filtering year from 2016 to 2025 and delete rows with missing neighbourhood value
toronto_clean = toronto_clean[
    (toronto_clean['year'] >= 2016) & (toronto_clean['year'] <= 2025) &
    toronto_clean['neighbourhood'].notnull()
]
toronto_clean = toronto_clean[toronto_clean['neighbourhood'] != 'NSA']

# Remove rows lat/lon values = 0 (location is hidden)
toronto_clean = toronto_clean[(toronto_clean['latitude'] != 0) & (toronto_clean['longitude'] != 0)]

# Fill small missing values
toronto_clean['premise_type'] = toronto_clean['premise_type'].fillna('Unknown')
toronto_clean['location_type'] = toronto_clean['location_type'].fillna('Unknown')

# Add city column
toronto_clean['city'] = 'Toronto'

# Convet year data type to int
toronto_clean['year'] = toronto_clean['year'].astype(int)
toronto_clean['hour'] = toronto_clean['hour'].astype(int)

# Remove duplicate rows
print("Duplicates rows: ", toronto_clean.duplicated().sum())
toronto_clean = toronto_clean.drop_duplicates()

# Reset index
toronto_clean = toronto_clean.reset_index(drop=True)

# Check cleaned Toronto data
print("Toronto cleaned: ", toronto_clean.shape)
print(toronto_clean.isnull().sum())
print("Crime type distribution: ", toronto_clean['crime_type'].value_counts())
print(toronto_clean.head())
toronto_clean.to_csv("../data/toronto/toronto_clean.csv", index=False)
print("Toronto cleaned data saved to ../data/toronto/toronto_clean.csv")

Duplicates rows:  31289
Toronto cleaned:  (360344, 12)
year             0
month            0
day_of_week      0
hour             0
offence          0
crime_type       0
neighbourhood    0
premise_type     0
location_type    0
latitude         0
longitude        0
city             0
dtype: int64
Crime type distribution:  crime_type
Assault            190602
Break and Enter     67325
Auto Theft          61754
Robbery             26804
Theft Over          13859
Name: count, dtype: int64
   year    month day_of_week  hour                      offence crime_type  \
0  2016  January  Friday         1          Assault With Weapon    Assault   
1  2016  January  Friday         1          Assault With Weapon    Assault   
2  2016  January  Friday         2          Assault With Weapon    Assault   
3  2016  January  Friday         1  Administering Noxious Thing    Assault   
4  2016  January  Friday         1                      Assault    Assault   

                neighbourhood premise_type

In [23]:
# Vancouver - Data cleaning

# Compute neigborhoods centroids from rows has valid coordinates
centroids = vancouver[vancouver['X'] != 0][['NEIGHBOURHOOD', 'X', 'Y']].groupby('NEIGHBOURHOOD').mean().reset_index().rename(columns={'NEIGHBOURHOOD': 'neighbourhood', 'X': 'x_fill', 'Y': 'y_fill'})
print("Centroids computed: ", centroids.shape)
import math

def utm_to_latlon(easting, northing, zone=10):
    a = 6378137.0 
    f = 1 / 298.257223563
    b = a * (1 - f)
    e2 = 1 - (b / a) ** 2
    e = math.sqrt(e2)
    k0 = 0.9996
    E0 = 500000.0
    lon0 = math.radians((zone - 1) * 6 - 180 + 3)

    M = northing / k0
    mu = M / (a * (1 - e2/4 - 3*e2**2/64 - 5*e2**3/256))

    e1 = (1 - math.sqrt(1 - e2)) / (1 + math.sqrt(1 - e2))
    phi1 = mu + (3*e1/2 - 27*e1**3/32) * math.sin(2*mu)
    phi1 += (21*e1**2/16 - 55*e1**4/32) * math.sin(4*mu)
    phi1 += (151*e1**3/96) * math.sin(6*mu)

    N1 = a / math.sqrt(1 - e2 * math.sin(phi1)**2)
    T1 = math.tan(phi1) ** 2
    C1 = e2 / (1 - e2) * math.cos(phi1) ** 2
    R1 = a * (1 - e2) / (1 - e2 * math.sin(phi1)**2) ** 1.5
    D = (easting - E0) / (N1 * k0)

    lat = phi1 - (N1 * math.tan(phi1) / R1) * (
        D**2/2 - (5 + 3*T1 + 10*C1 - 4*C1**2 - 9*e2/(1-e2)) * D**4/24
    )
    lon = lon0 + (D - (1 + 2*T1 + C1) * D**3/6) / math.cos(phi1)

    return math.degrees(lat), math.degrees(lon)

month_mapping = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April',
    5: 'May', 6: 'June', 7: 'July', 8: 'August',
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}

crime_type_mapping = {
    'Theft from Vehicle': 'Theft',
    'Theft of Vehicle': 'Auto Theft',
    'Other Theft': 'Theft',
    'Break and Enter Commercial': 'Break and Enter',
    'Break and Enter Residential/Other': 'Break and Enter',
    'Mischief': 'Mischief',
    'Vehicle Collision or Pedestrian Struck (with Fatality)': 'Other',
    'Vehicle Collision or Pedestrian Struck (with Injury)': 'Other',
    'Offence Against a Person': 'Assault',
    'Homicide': 'Assault',
}

vancouver_clean = vancouver[[
    'YEAR', 'MONTH', 'DAY', 'HOUR',
    'TYPE', 'NEIGHBOURHOOD', 'X', 'Y',
]].copy()

vancouver_clean.columns = [
    'year', 'month', 'day', 'hour',
    'crime_type', 'neighbourhood', 'x_utm', 'y_utm'
]

# Clean crime type
vancouver_clean['crime_type'] = (
    vancouver_clean['crime_type'].str.strip()
    .replace(crime_type_mapping)
)

# Filter years
vancouver_clean = vancouver_clean[
    (vancouver_clean['year'] >= 2016) & (vancouver_clean['year'] <= 2025)
]

# Remove missing neighbourhoods
vancouver_clean = vancouver_clean[
    vancouver_clean['neighbourhood'].notna() &
    (vancouver_clean['neighbourhood'].str.strip() != '')
]

# Fill X = 0, Y = 0 with neighbourhood centroids
vancouver_clean = vancouver_clean.merge(centroids, on='neighbourhood', how='left')

vancouver_clean['x_utm'] = vancouver_clean.apply(
    lambda r: r['x_fill'] if r['x_utm'] == 0 else r['x_utm'], axis=1
)
vancouver_clean['y_utm'] = vancouver_clean.apply(
    lambda r: r['y_fill'] if r['y_utm'] == 0 else r['y_utm'], axis=1
)
vancouver_clean = vancouver_clean.drop(columns=['x_fill', 'y_fill'])

# Remove rows with x = 0 or y = 0
vancouver_clean = vancouver_clean[
    (vancouver_clean['x_utm'] != 0) & (vancouver_clean['y_utm'] != 0)
]

# Convert UTM to lat/lon
print("Converting coordinates (may take ~30 seconds)...")
coords = vancouver_clean.apply(
    lambda r: utm_to_latlon(r['x_utm'], r['y_utm']), axis=1
)
vancouver_clean['latitude']  = coords.apply(lambda x: x[0])
vancouver_clean['longitude'] = coords.apply(lambda x: x[1])
vancouver_clean = vancouver_clean.drop(columns=['x_utm', 'y_utm'])

# Verify lat, lon conversion
print("Coordinate check:")
print(vancouver_clean[['latitude','longitude']].describe())

# Map month before drop_duplicates
vancouver_clean['month'] = vancouver_clean['month'].map(month_mapping)

# Add missing columns for consistency
vancouver_clean['city'] = 'Vancouver'
vancouver_clean['offence'] = vancouver_clean['crime_type']
vancouver_clean['premise_type'] = 'Unknown'
vancouver_clean['location_type'] = 'Unknown'

# Remove duplicate
print("Duplicates rows: ", vancouver_clean.duplicated().sum())
vancouver_clean = vancouver_clean.drop_duplicates().reset_index(drop=True)

# Check cleaned Vancouver data
print("Vancouver cleaned: ", vancouver_clean.shape)
print(vancouver_clean['crime_type'].value_counts())
print(vancouver_clean.isnull().sum())
print(vancouver_clean.head())
vancouver_clean.to_csv("../data/vancouver/vancouver_clean.csv", index=False)
print("Vancouver cleaned data saved to ../data/vancouver/vancouver_clean.csv")


Centroids computed:  (24, 3)
Converting coordinates (may take ~30 seconds)...
Coordinate check:
            latitude      longitude
count  352320.000000  352320.000000
mean       49.266109    -123.106072
std         0.021367       0.034139
min        49.200910    -123.224021
25%        49.258164    -123.124641
50%        49.275095    -123.112796
75%        49.281816    -123.090692
max        49.313349    -122.835318
Duplicates rows:  15970
Vancouver cleaned:  (336350, 12)
crime_type
Theft               197886
Mischief             50049
Break and Enter      35380
Assault              17394
Theft of Bicycle     15373
Other                11201
Auto Theft            9067
Name: count, dtype: int64
year             0
month            0
day              0
hour             0
crime_type       0
neighbourhood    0
latitude         0
longitude        0
city             0
offence          0
premise_type     0
location_type    0
dtype: int64
   year      month  day  hour       crime_type          

In [33]:
# Merge datasets
common_cols = ['city', 'year', 'month', 'hour', 'crime_type', 'neighbourhood', 'premise_type', 'latitude', 'longitude']

toronto_final = toronto_clean[common_cols].copy()
vancouver_final = vancouver_clean[common_cols].copy()

# Verify before concat
print("Toronto Assault:", toronto_final[toronto_final['crime_type'] == 'Assault'].shape[0])
print("Vancouver Assault:", vancouver_final[vancouver_final['crime_type'] == 'Assault'].shape[0])

# Concat
df = pd.concat([toronto_final, vancouver_final], ignore_index=True)
print("\nAfter concat:", df.shape)
print("Assault total:", df[df['crime_type'] == 'Assault'].shape[0])



# Standardize crime groups
crime_group_mapping = {
    'Assault': 'Violent Crime',
    'Robbery': 'Violent Crime',

    'Auto Theft': 'Property Crime',
    'Break and Enter': 'Property Crime',
    'Theft': 'Property Crime',
    'Theft Over': 'Property Crime',
    'Theft of Bicycle': 'Property Crime',

    'Mischief': 'Property Damage',

    'Other': 'Other'
}

df['crime_group'] = (
    df['crime_type']
    .map(crime_group_mapping)
    .fillna('Other')
)

# Order months
month_order = [
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
]

df['month'] = pd.Categorical(
    df['month'],
    categories=month_order,
    ordered=True
)

# Optimize datatypes
df['crime_type'] = df['crime_type'].astype('category')
df['crime_group'] = df['crime_group'].astype('category')
df['city'] = df['city'].astype('category')

# Validation
print("\nRows per city:", df['city'].value_counts())
print("\nYears available:", df.groupby('city')['year'].agg(['min', 'max']))
print("\nCrime types:", df.groupby(['city', 'crime_type']).size().unstack(fill_value=0))
print("\nCrime group distribution:", df.groupby(['city', 'crime_group']).size().unstack(fill_value=0))
print("\nMissing values:", df.isnull().sum())
print("\nCoordinates sample:")
print(df.groupby('city')[['latitude','longitude']].mean())

# Save final dataset
df.to_csv("../data/merged_crime_data.csv", index=False)

df.to_parquet(
    "../data/merged_crime_data.parquet",
    index=False
)

print("Saved successfully: ", df.shape)

Toronto Assault: 190602
Vancouver Assault: 17394

After concat: (696694, 9)
Assault total: 207996

Rows per city: city
Toronto      360344
Vancouver    336350
Name: count, dtype: int64

Years available:             min   max
city                 
Toronto    2016  2025
Vancouver  2016  2025

Crime types: crime_type  Assault  Auto Theft  Break and Enter  Mischief  Other  Robbery  \
city                                                                         
Toronto      190602       61754            67325         0      0    26804   
Vancouver     17394        9067            35380     50049  11201        0   

crime_type   Theft  Theft Over  Theft of Bicycle  
city                                              
Toronto          0       13859                 0  
Vancouver   197886           0             15373  

Crime group distribution: crime_group  Other  Property Crime  Property Damage  Violent Crime
city                                                              
Toronto          